# Gold Layer Dataframes

This notebook shows the dataframes of our gold layer in order to better understand it and know if there are any problems with our data ( keep in mind here we ingest from the bronze layer so we won't have quality data ).

Thus here we won't define any primary keys, foreign keys or other constraints yet since we need actual tables for that. However we can still run sql queries on the dataframes by turning them to views.


##The date dimension

In [0]:
from pyspark.sql.functions import col, to_date, date_format, year, month, dayofmonth, dayofweek, when, hour, minute

In [0]:
df_orders = spark.read.table("workspace.bronze_restaurant.orders")
df_weather = spark.read.table("workspace.bronze_restaurant.weather")

dates_orders = df_orders.select(to_date(col("timestamp")).alias("full_date"))
dates_weather = df_weather.select(to_date(col("timestamp")).alias("full_date"))

unique_dates = dates_orders.union(dates_weather).distinct().dropna()

dim_date = unique_dates.select(
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_id"),
    col("full_date"),
    year(col("full_date")).alias("year"),
    month(col("full_date")).alias("month_number"),
    dayofmonth(col("full_date")).alias("day"),
    date_format(col("full_date"), "MMMM").alias("month_name"),
    dayofweek(col("full_date")).alias("day_of_week"),
    date_format(col("full_date"), "EEEE").alias("day_name_of_week")
).withColumn(
    "is_weekend", 
    when(col("day_of_week").isin([1, 7]), True).otherwise(False) 
)

display(dim_date)

##The time dimension

In [0]:
time_orders = df_orders.select(
    date_format(col("timestamp"), "HHmm").cast("int").alias("time_id"), # we can remove the cast to int if we want to keep it more comprehensible
    date_format(col("timestamp"), "HH:mm").alias("time_label"),
    hour(col("timestamp")).alias("hour"),
    minute(col("timestamp")).alias("minute")
)
time_weather = df_weather.select(
    date_format(col("timestamp"), "HHmm").cast("int").alias("time_id"),
    date_format(col("timestamp"), "HH:mm").alias("time_label"),
    hour(col("timestamp")).alias("hour"),
    minute(col("timestamp")).alias("minute")
)
unique_times = time_orders.union(time_weather).distinct().dropna()

dim_time = unique_times.withColumn(
    "day_time",
    when((col("hour") >= 6) & (col("hour") < 12), "Morning")
    .when((col("hour") >= 12) & (col("hour") < 17), "Afternoon")
    .when((col("hour") >= 17) & (col("hour") < 21), "Evening")
    .otherwise("Night") 
).orderBy("time_id")

display(dim_time)

##The menu dimension

In [0]:
dim_menu = spark.read.table("workspace.bronze_restaurant.menu")
dim_menu = dim_menu.drop("ingestion_timestamp")
dim_menu = dim_menu.drop("source_system")
dim_menu.display()

##The employees dimension

In [0]:
dim_employees = spark.read.table("workspace.bronze_restaurant.employees")
dim_employees = dim_employees.drop("ingestion_timestamp")
dim_employees = dim_employees.drop("source_system")
dim_employees.display()

##The weather dimension

Here there are 2 points we'll need to discuss : 

First : in the orders table we have data that come from the future hhhhh ( orders through the whole year), but in the weather table we have data about the first 3 months, HOWEVER in the orders table each row have a linked weather_id even if the the corresponding date doesn't exist in the weather table so do we leave it as it is ?

Second : in the weather table there is the temperature attribute which is a "continuous" measure and we said that we'll want to turn it into intervals, HOWEVER we already have the weather_ids in the orders tables, and if we transformed the temperature attribute we'll have to make new ids and replace the ones in the orders tables with them ( because if we make intevals we're gonna have way fewer rows in the weather table so news ids), so can we just not do that ?

Alkholassa : la kan momkin nkhaliw dim_weather kima hia f csv file hhhhhh.

In [0]:
dim_weather = spark.read.table("workspace.bronze_restaurant.weather")
dim_weather = dim_weather.drop("ingestion_timestamp")
dim_weather = dim_weather.drop("source_system")
dim_weather = dim_weather.drop("timestamp")
dim_weather.display()

##The sales fact 

In [0]:
df_orders = spark.read.table("workspace.bronze_restaurant.orders")
df_menu = spark.read.table("workspace.bronze_restaurant.menu")
df_orders = df_orders.drop("ingestion_timestamp")
df_orders = df_orders.drop("source_system")
df_orders = df_orders.withColumn("date_id", date_format(col("timestamp"), "yyyyMMdd").cast("int"))
df_orders = df_orders.withColumn("time_id", date_format(col("timestamp"), "HHmm").cast("int"))
df_orders = df_orders.drop("timestamp")

df_details = spark.read.table("workspace.bronze_restaurant.order_details")
df_details = df_details.drop("ingestion_timestamp")
df_details = df_details.drop("source_system")

fact_sales = ( 
            df_details
            .join(df_orders, on = ["order_id"])
            .join(df_menu, on = ["item_id"])
            .select(
                "order_id",
                "detail_id",
                "item_id",
                "server_id",
                "weather_id",
                "date_id",
                "time_id",
                "order_type",
                "qty",
                (col("qty") * col("price")).alias("total_amount")
            )

)
fact_sales.display()

##Defining views

In [0]:
dim_date.createOrReplaceTempView("temp_date")
dim_time.createOrReplaceTempView("temp_time")
dim_employees.createOrReplaceTempView("temp_emp")
dim_menu.createOrReplaceTempView("temp_menu")
dim_weather.createOrReplaceTempView("temp_weather")
fact_sales.createOrReplaceTempView("temp_sales")

##Random queries

In [0]:
%sql
-- What is the revenu per item per day time
SELECT 
    m.name,
    t.day_time,
    SUM(f.total_amount) as total_revenue
FROM temp_sales f
JOIN temp_menu m ON f.item_id = m.item_id
JOIN temp_time t ON f.time_id = t.time_id
GROUP BY m.name, t.day_time
ORDER BY total_revenue;

In [0]:
%sql
-- Does the weather affect the number of orders ?
SELECT 
    w.condition,
    COUNT(DISTINCT f.order_id) as number_of_orders,
    AVG(f.qty) as avg_items_per_order
FROM temp_sales f
JOIN temp_weather w ON f.weather_id = w.weather_id
GROUP BY w.condition;

I believe counting orders of an order_type by the weather is more interesting. One issue tho, it's always equally split...

In [0]:
%sql
SELECT 
    w.condition,
    f.order_type,
    COUNT(DISTINCT f.order_id) as number_of_orders,
    AVG(f.qty) as avg_items_per_order
FROM temp_sales f
JOIN temp_weather w ON f.weather_id = w.weather_id
GROUP BY w.condition, f.order_type
Order BY w.condition;

In [0]:
%sql
-- How much money did we make in each day of the week through the year ?
SELECT 
    d.day_of_week,
    SUM(total_amount) as revenue
FROM temp_sales f
JOIN temp_date d ON f.date_id = d.date_id
JOIN temp_menu m ON f.item_id = m.item_id
GROUP BY d.day_of_week, d.day_name_of_week
ORDER BY d.day_of_week;

In [0]:
%sql
-- How many orders did we have per hour, per day ?
SELECT 
    d.full_date, 
    t.hour, 
    COUNT(DISTINCT f.order_id) AS number_of_orders_per_hour 
FROM temp_sales f
JOIN temp_time t ON f.time_id = t.time_id
JOIN temp_date d ON f.date_id = d.date_id
GROUP BY d.full_date, t.hour
ORDER BY d.full_date, t.hour;

In [0]:
%sql
-- How much did we make per order
SELECT
    order_id,
    sum(total_amount) as total_revenue_per_order 
FROM temp_sales 
GROUP BY order_id 
ORDER BY order_id

More random queries lol :

In [0]:
%sql
-- How much revenue was every waiter responsible of :
SELECT
    sum(total_amount) as total_revenue_per_waiter,
    server_id,
    e.name
FROM temp_sales s
JOIN temp_emp e ON s.server_id = e.emp_id
GROUP BY server_id, name 
ORDER BY total_revenue_per_waiter desc;

In [0]:
%sql
-- What are the most popular items (based on how many times they were ordered)
SELECT 
    m.name,
    count(f.detail_id) as number_of_orders
FROM temp_sales f
JOIN temp_menu m ON f.item_id = m.item_id
GROUP BY m.name
ORDER BY number_of_orders DESC;
